# 08 - Spatial Concentration & Hotspot Sanity Diagnostics

**Title**: `08_hotspot_sanity.ipynb`  
**Purpose**: Evaluate geographic incident concentration, address normalization distributions, spatial-temporal weak clustering, projection origin proximity, and coordinate capping retention profiles across all complaint groups in Brooklyn.  
**Inputs**: `data/processed/complaints_clean.csv`, `data/processed/projection.json`  
**Outputs**: Top-10 coordinate and address rankings, high-volume threshold concentration tables, coordinate percentiles, weak cluster chain analysis, largest cluster weekly trajectory, origin proximity verification, effective row capping retention table, diagnostic log-log distribution and hotspot spatial figures.  
**Runtime**: ~15-25 seconds

In [1]:
import sys; from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists()); sys.path.insert(0, str(ROOT))
from src.nbutils import Checker, md
RUN_HEAVY = False
RUN_API = False

In [2]:
import numpy as np
import pandas as pd
from src.preprocessing.hotspot_sanity import (
    load_clean_and_projection,
    compute_top_coordinates,
    compute_top_addresses,
    compute_coordinate_concentration,
    compute_percentiles_per_group,
    compute_weak_clusters,
    compute_largest_cluster_weekly,
    compute_origin_proximity,
    compute_effective_rows_capping,
    plot_weekly_largest_cluster,
    plot_coordinate_histograms,
    plot_noise_top300_scatter,
    GROUPS,
)

clean_path = ROOT / "data/processed/complaints_clean.csv"
proj_path = ROOT / "data/processed/projection.json"

print(f"Loading cleaned dataset from {clean_path.relative_to(ROOT)}...")
df, proj_info = load_clean_and_projection(clean_path, proj_path)
print(f"Loaded {len(df):,} cleaned records. Projection origin: lat0={proj_info['lat0']}, lon0={proj_info['lon0']}")

Loading cleaned dataset from data\processed\complaints_clean.csv...


Loaded 236,507 cleaned records. Projection origin: lat0=40.661641, lon0=-73.952455


In [3]:
print("=== Section 1: Top Coordinates & Address Normalization ===")
top_coords = compute_top_coordinates(df, n=10)
top_addrs = compute_top_addresses(df, n=10)

print("\n--- Top 10 (lat, lon) Pairs Overall ---")
print(top_coords["overall"].to_string(index=False))

for g in GROUPS:
    print(f"\n--- Top 10 (lat, lon) Pairs for Group: {g} ---")
    print(top_coords[g].to_string(index=False))

print("\n--- Top 10 Normalized Addresses Overall ---")
print(top_addrs["overall"].to_string(index=False))

for g in GROUPS:
    print(f"\n--- Top 10 Normalized Addresses for Group: {g} ---")
    print(top_addrs[g].to_string(index=False))

=== Section 1: Top Coordinates & Address Normalization ===



--- Top 10 (lat, lon) Pairs Overall ---
  group  rank       lat        lon  rows  share_pct  distinct_address_norm         top_address
overall     1 40.608101 -73.999384  8844   3.739424                      1      1869 83 STREET
overall     2 40.699800 -73.983616  1905   0.805473                      2     177 SAND STREET
overall     3 40.651624 -73.960235  1343   0.567848                      2  111 EAST 21 STREET
overall     4 40.695482 -73.980649  1182   0.499774                      1        85 NAVY WALK
overall     5 40.679775 -73.908893  1140   0.482015                      1     203 HULL STREET
overall     6 40.674188 -73.967832   969   0.409713                      4  295 ST JOHNS PLACE
overall     7 40.704396 -73.963991   876   0.370391                      1    75 WILSON STREET
overall     8 40.577620 -73.990378   729   0.308236                      2 2832 WEST 23 STREET
overall     9 40.591350 -73.967530   661   0.279484                      3  2411 EAST 3 STREET
overall  

In [4]:
print("=== Section 2: Coordinate Concentration & Percentiles ===")
df_conc = compute_coordinate_concentration(df)
df_percentiles = compute_percentiles_per_group(df)

print("\n--- Coordinate Concentration Above Volume Thresholds (>50, >200, >1000 rows) ---")
print(df_conc.to_string(index=False))

print("\n--- Rows per Distinct Coordinate Percentiles ---")
print(df_percentiles.to_string(index=False))

=== Section 2: Coordinate Concentration & Percentiles ===



--- Coordinate Concentration Above Volume Thresholds (>50, >200, >1000 rows) ---
       group threshold  coords_within_group  rows_within_group  share_within_group_pct  coords_overall_dataset  rows_overall_dataset  share_overall_dataset_pct
       flood       >50                    2                198                3.586957                      21                   242                   4.384058
       flood      >200                    0                  0                0.000000                       3                     6                   0.108696
       flood     >1000                    0                  0                0.000000                       0                     0                   0.000000
       noise       >50                  424              65757               31.040440                     432                 65909                  31.112192
       noise      >200                   58              33460               15.794716                      58        

In [5]:
print("=== Section 3: Chained Weak Clusters & Largest Cluster Trajectory ===")
df_top5_clusters, largest_cluster, large_rows = compute_weak_clusters(df, gap_hours=24.0)
df_weekly = compute_largest_cluster_weekly(large_rows)

print("\n--- Top 5 Chained Weak Clusters per Group ---")
print(df_top5_clusters.to_string(index=False))

print(f"\n--- Largest Weak Cluster Overall ---")
print(f"Group: {largest_cluster['group']}, Address: {largest_cluster['address_norm']}, Complaint Type: {largest_cluster['complaint_type']}")
print(f"Size: {largest_cluster['size']:,} complaints, Span: {largest_cluster['span_days']:.2f} days, Distinct Coords: {largest_cluster['distinct_coords']}")

print("\n--- Weekly Complaint Trajectory of Largest Weak Cluster (First 10 Weeks) ---")
print(df_weekly.head(10).to_string(index=False))

=== Section 3: Chained Weak Clusters & Largest Cluster Trajectory ===



--- Top 5 Chained Weak Clusters per Group ---
       group              address_norm      complaint_type  cluster_id  size     first_timestamp      last_timestamp  distinct_coords  span_days  group_rank
       flood            EAST 15 STREET   Sewer Maintenance           8    11 2026-08-24 11:30:07 2026-08-27 10:41:46                1   2.966424           1
       flood            EAST 15 STREET   Sewer Maintenance           2     9 2026-07-29 06:36:00 2026-08-02 14:32:49                1   4.331123           2
       flood             1702 AVENUE Z               Sewer           5     8 2026-05-24 13:56:00 2026-05-26 19:20:00                1   2.225000           3
       flood            EAST 15 STREET   Sewer Maintenance           5     8 2026-08-13 14:26:52 2026-08-15 19:19:33                1   2.203252           4
       flood  443 KNICKERBOCKER AVENUE               Sewer           1     7 2026-05-20 22:07:00 2026-05-20 23:04:00                1   0.039583           5
       nois

In [6]:
print("=== Section 4: Projection Origin Proximity & Effective Row Table ===")
df_origin, min_dist_overall = compute_origin_proximity(df, proj_info)
df_effective = compute_effective_rows_capping(df, caps=[10, 25, 50, 100])

print("\n--- Rows Within 20 m of Projection Origin ---")
print(df_origin.to_string(index=False))
print(f"Overall Minimum Distance to Projection Origin: {min_dist_overall:.2f} m")

print("\n--- Effective Row Retention Table Under Coordinate Capping (Step 3 Planning) ---")
print(df_effective.to_string(index=False))

=== Section 4: Projection Origin Proximity & Effective Row Table ===



--- Rows Within 20 m of Projection Origin ---
       group  rows_within_20m  min_dist_m
       flood                0  128.734967
       noise                0   20.661931
        road                0  101.926468
sewer_backup                0  128.734967
     overall                0   20.661931
Overall Minimum Distance to Projection Origin: 20.66 m

--- Effective Row Retention Table Under Coordinate Capping (Step 3 Planning) ---
       group  total_rows  cap_10  cap_25  cap_50  cap_100
       flood        5520    5231    5368    5422     5482
       noise      211843  121328  149966  167286   180695
        road       16138   15014   15849   16117    16138
sewer_backup        3006    2999    3006    3006     3006


In [7]:
fig1_path = ROOT / "report/figures/08_weak_cluster_weekly.png"
plot_weekly_largest_cluster(df_weekly, fig1_path)
print(f"Saved figure: {fig1_path.relative_to(ROOT)}")

fig2_path = ROOT / "report/figures/08_rows_per_coord_hist.png"
plot_coordinate_histograms(df, fig2_path)
print(f"Saved figure: {fig2_path.relative_to(ROOT)}")

fig3_path = ROOT / "report/figures/08_noise_top300_scatter.png"
plot_noise_top300_scatter(df, fig3_path)
print(f"Saved figure: {fig3_path.relative_to(ROOT)}")

Saved figure: report\figures\08_weak_cluster_weekly.png


Saved figure: report\figures\08_rows_per_coord_hist.png


Saved figure: report\figures\08_noise_top300_scatter.png


In [8]:
checker = Checker()

# 1. top coordinate overall rows 8844 and its address_norm '1869 83 STREET'
top_row = top_coords["overall"].iloc[0]
checker.check("top coordinate overall rows = 8844", int(top_row["rows"]), 8844)
checker.check("top coordinate overall address = '1869 83 STREET'", str(top_row["top_address"]), "1869 83 STREET")

# 2. distinct coordinates overall 56878
checker.check("distinct coordinates overall = 56878", int(df_percentiles[df_percentiles["group"] == "overall"]["distinct_coords"].values[0]), 56878)

# 3. noise coordinates above 50 rows: 424 coordinates holding 65757 rows; above 200: 58 and 33460; above 1000: 5 and 14413
c50 = df_conc[(df_conc["group"] == "noise") & (df_conc["threshold"] == ">50")].iloc[0]
c200 = df_conc[(df_conc["group"] == "noise") & (df_conc["threshold"] == ">200")].iloc[0]
c1000 = df_conc[(df_conc["group"] == "noise") & (df_conc["threshold"] == ">1000")].iloc[0]

checker.check("noise coordinates > 50 count = 424", int(c50["coords_within_group"]), 424)
checker.check("noise coordinates > 50 rows = 65757", int(c50["rows_within_group"]), 65757)
checker.check("noise coordinates > 200 count = 58", int(c200["coords_within_group"]), 58)
checker.check("noise coordinates > 200 rows = 33460", int(c200["rows_within_group"]), 33460)
checker.check("noise coordinates > 1000 count = 5", int(c1000["coords_within_group"]), 5)
checker.check("noise coordinates > 1000 rows = 14413", int(c1000["rows_within_group"]), 14413)

# 4. rows per coordinate max per group flood 138, noise 8844, road 70, sewer_backup 15
p_map = df_percentiles.set_index("group")
checker.check("rows per coordinate max flood = 138", int(p_map.loc["flood", "max"]), 138)
checker.check("rows per coordinate max noise = 8844", int(p_map.loc["noise", "max"]), 8844)
checker.check("rows per coordinate max road = 70", int(p_map.loc["road", "max"]), 70)
checker.check("rows per coordinate max sewer_backup = 15", int(p_map.loc["sewer_backup", "max"]), 15)

# 5. noise p99 = 47
checker.check("noise p99 = 47", int(p_map.loc["noise", "p99"]), 47)

# 6. empty address share road = 32.10 (tol 0.01)
road_empty_share = round(float((df[df["group"] == "road"]["address_norm"].str.strip() == "").mean()) * 100.0, 2)
checker.check("empty address share road = 32.10", road_empty_share, 32.10, tol=0.01)

# 7. largest weak cluster size 8843 with span 364.46 days (tol 0.01)
checker.check("largest weak cluster size = 8843", int(largest_cluster["size"]), 8843)
checker.check("largest weak cluster span = 364.46 days", round(float(largest_cluster["span_days"]), 2), 364.46, tol=0.01)

# 8. its first week (week starting 2025-10-28) count 31 and its peak weekly count 274
w_first = int(df_weekly[df_weekly["week_start_str"] == "2025-10-28"]["count"].values[0])
w_peak = int(df_weekly["count"].max())
checker.check("largest weak cluster first week count = 31", w_first, 31)
checker.check("largest weak cluster peak weekly count = 274", w_peak, 274)

# 9. minimum distance to the origin overall 20.66 m (tol 0.01)
checker.check("minimum distance to origin overall = 20.66 m", round(min_dist_overall, 2), 20.66, tol=0.01)

checker.summary()

[PASS] top coordinate overall rows = 8844: actual=8844, expected=8844
[PASS] top coordinate overall address = '1869 83 STREET': actual=1869 83 STREET, expected=1869 83 STREET
[PASS] distinct coordinates overall = 56878: actual=56878, expected=56878
[PASS] noise coordinates > 50 count = 424: actual=424, expected=424
[PASS] noise coordinates > 50 rows = 65757: actual=65757, expected=65757
[PASS] noise coordinates > 200 count = 58: actual=58, expected=58
[PASS] noise coordinates > 200 rows = 33460: actual=33460, expected=33460
[PASS] noise coordinates > 1000 count = 5: actual=5, expected=5
[PASS] noise coordinates > 1000 rows = 14413: actual=14413, expected=14413
[PASS] rows per coordinate max flood = 138: actual=138, expected=138
[PASS] rows per coordinate max noise = 8844: actual=8844, expected=8844
[PASS] rows per coordinate max road = 70: actual=70, expected=70
[PASS] rows per coordinate max sewer_backup = 15: actual=15, expected=15
[PASS] noise p99 = 47: actual=47, expected=47
[PASS]

In [9]:
findings_08 = f"""### Findings
- **Distinct Coordinate Footprint**: Across the 236,507 cleaned complaint records in Brooklyn, incidents resolve to {int(df_percentiles[df_percentiles['group'] == 'overall']['distinct_coords'].values[0]):,} distinct geographic (lat, lon) pairs.
- **Extreme Noise Hotspot Concentration**: The single most heavily reported coordinate in Brooklyn ({top_row['lat']}, {top_row['lon']}) accounts for {int(top_row['rows']):,} complaints ({top_row['share_pct']:.2f}% of the borough-wide dataset), all attributed to the normalized address `{top_row['top_address']}`.
- **Threshold Concentration Profiles**: High-volume coordinate clustering is heavily concentrated within the noise group:
  - {int(c50['coords_within_group'])} coordinates have >50 complaints, accounting for {int(c50['rows_within_group']):,} records ({c50['share_within_group_pct']:.2f}% of all noise).
  - {int(c200['coords_within_group'])} coordinates have >200 complaints, accounting for {int(c200['rows_within_group']):,} records ({c200['share_within_group_pct']:.2f}% of all noise).
  - {int(c1000['coords_within_group'])} coordinates have >1000 complaints, accounting for {int(c1000['rows_within_group']):,} records ({c1000['share_within_group_pct']:.2f}% of all noise).
- **Group Maximum Volume Contrasts**: Maximum complaints at any single coordinate vary drastically by domain: {int(p_map.loc['noise', 'max']):,} for noise, {int(p_map.loc['flood', 'max'])} for flood, {int(p_map.loc['road', 'max'])} for road, and {int(p_map.loc['sewer_backup', 'max'])} for sewer backup. The 99th percentile coordinate count for noise is {int(p_map.loc['noise', 'p99'])}.
- **Missing Address Structure**: Incident geocoding retains coordinates even when specific street addresses are unrecorded; the empty address share reaches {road_empty_share:.2f}% in the road condition group.
- **Chained Weak Cluster Persistence**: The single largest chained weak cluster (key: group, address, complaint type; gap <= 24 hours) contains {int(largest_cluster['size']):,} complaints spanning {float(largest_cluster['span_days']):.2f} days across {int(largest_cluster['distinct_coords'])} distinct coordinates. Its initial week (commencing 2025-10-28) logged {w_first} complaints, reaching a peak weekly volume of {w_peak} complaints.
- **Projection Centroid Sanity**: No complaints fall within 20 meters of the equirectangular projection origin (lat0={proj_info['lat0']}, lon0={proj_info['lon0']}), with the closest recorded complaint located {min_dist_overall:.2f} meters away, confirming that the origin point does not artificially distort incident coordinates.
- **Coordinate Capping Sensitivity**: Under a candidate cap of 50 complaints per coordinate for future spatial modeling, noise records decrease from {int(df_effective[df_effective['group'] == 'noise']['total_rows'].values[0]):,} to {int(df_effective[df_effective['group'] == 'noise']['cap_50'].values[0]):,}, while flood retains {int(df_effective[df_effective['group'] == 'flood']['cap_50'].values[0]):,} of its {int(df_effective[df_effective['group'] == 'flood']['total_rows'].values[0]):,} records ({float(df_effective[df_effective['group'] == 'flood']['cap_50'].values[0] / df_effective[df_effective['group'] == 'flood']['total_rows'].values[0] * 100.0):.2f}%).
"""
md(findings_08)
checker.summary()

### Findings
- **Distinct Coordinate Footprint**: Across the 236,507 cleaned complaint records in Brooklyn, incidents resolve to 56,878 distinct geographic (lat, lon) pairs.
- **Extreme Noise Hotspot Concentration**: The single most heavily reported coordinate in Brooklyn (40.60810108365, -73.99938414563) accounts for 8,844 complaints (3.74% of the borough-wide dataset), all attributed to the normalized address `1869 83 STREET`.
- **Threshold Concentration Profiles**: High-volume coordinate clustering is heavily concentrated within the noise group:
  - 424 coordinates have >50 complaints, accounting for 65,757 records (31.04% of all noise).
  - 58 coordinates have >200 complaints, accounting for 33,460 records (15.79% of all noise).
  - 5 coordinates have >1000 complaints, accounting for 14,413 records (6.80% of all noise).
- **Group Maximum Volume Contrasts**: Maximum complaints at any single coordinate vary drastically by domain: 8,844 for noise, 138 for flood, 70 for road, and 15 for sewer backup. The 99th percentile coordinate count for noise is 47.
- **Missing Address Structure**: Incident geocoding retains coordinates even when specific street addresses are unrecorded; the empty address share reaches 32.10% in the road condition group.
- **Chained Weak Cluster Persistence**: The single largest chained weak cluster (key: group, address, complaint type; gap <= 24 hours) contains 8,843 complaints spanning 364.46 days across 1 distinct coordinates. Its initial week (commencing 2025-10-28) logged 31 complaints, reaching a peak weekly volume of 274 complaints.
- **Projection Centroid Sanity**: No complaints fall within 20 meters of the equirectangular projection origin (lat0=40.661641, lon0=-73.952455), with the closest recorded complaint located 20.66 meters away, confirming that the origin point does not artificially distort incident coordinates.
- **Coordinate Capping Sensitivity**: Under a candidate cap of 50 complaints per coordinate for future spatial modeling, noise records decrease from 211,843 to 167,286, while flood retains 5,422 of its 5,520 records (98.22%).


CHECKS: 20 passed, 0 failed


### Limitations

- High-density spatial hotspots in citizen reporting datasets may reflect acute localized reporting tendencies or repeated automated submissions from specific premises rather than physical hazard intensity across the broader geographic zone.
- Geocoding resolution is bounded by municipal geocoding precision, where multiple physical doorways or adjacent structures are often assigned identical street-segment midpoints or parcel centroids.
- The 24-hour chaining threshold for weak clusters links complaints into contiguous temporal runs; incidents separated by 25 or more hours are partitioned into distinct clusters regardless of underlying persistent conditions.